## Core of the Transformer Model


for every head- Q,K,V -> dimension =64 

8 heads

combined dimension =64*8 =512

In [ ]:
"""
TRANSFORMER TEXT GENERATION - REORGANIZED FOR LEARNING
------------------------------------------------------------------------------
Better structure: Overview → Data → Architecture → Training → Generation
Preserving all the detailed understanding comments!
"""

import os
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.layers import Layer, Embedding, Dense, LayerNormalization, Dropout
import numpy as np


# SECTION 1: CONFIGURATION & HYPERPARAMETERS
# -------------------------------------------------------------------------------
# Put all important settings at the TOP so they're easy to find and modify

class Config:
    """All hyperparameters in one place - easy to experiment!"""
    # Data settings
    FILE_PATH = "../../data/harry_potter.txt"
    SEQ_LENGTH = 100  # context window size --> Means The model can only "remember" 100 previous tokens
    
    # Model architecture
    EMBED_DIM = 128      # Each token (word) becomes a vector of length 128. -> "harry" → [0.12, -0.98, 0.44, ..., 0.03]  (128 numbers)
                         # Small model → 64–256  GPT-3 → 12288 (very large)
    NUM_HEADS = 4        # Number of attention heads, Of course its multi-head attention and not single head attention 
                            # Example intuition:
                            # Head 1 → grammar
                            # Head 2 → long-range dependency
                            # Head 3 → entities
                            # Head 4 → local context
                            
                         # ✨each head will have (embed_dim / num_heads) dimensions ie. 128/4 = 32 dimensions
                         # ✨But good news is heads are concatenated back to embed_dim after attention i.e. 128
    FF_DIM = 512         # Feed-forward layer size , This is the hidden size of the feed-forward network inside each Transformer block.
                         # Think of it as: “After reading, how deeply do I think about each word?”
                         # Rule of thumb (IMPORTANT):ff_dim ≈ 4 × embed_dim
                         
    NUM_LAYERS = 1       # 🌟🌟 Number of Transformer layers (depth of the model)
                         # you can change this to 2, 4, 6, 8, etc. 🌟🌟
                         #  1 layer  → shallow thought
                         #  6 layers → structured reasoning
                         #  12 layers → deep abstraction
                         
    DROPOUT_RATE = 0.1   # Regularization
    
    """
        One Mental Model (remember this):

        embed_dim → what a word knows
        num_heads → how many ways it looks
        ff_dim → how deeply it thinks
        num_layers → how many times it thinks
        
    """
    
    # Training settings
    EPOCHS = 50
    BATCH_SIZE = 128
    VALIDATION_SPLIT = 0.1
    
    # Generation settings
    MODEL_SAVE_PATH = "../../models/harry_transformer_model.keras"


config = Config()






STEP 1: LOADING DATA
Dataset length: 457729 characters
Vocabulary size: 6663 unique words
input_sequences: 80922
 Data shape: X=(80922, 100), y=(80922,)

STEP 2: BUILDING MODEL
After embeddings: (None, 100, 128)
After transformer block 1: (None, 100, 128)
After taking last token: (None, 128)
Final output: (None, 6663)

✅ Model built successfully!
Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, 100)]             0         
                                                                 
 token_and_position_embeddin  (None, 100, 128)         865664    
 g_3 (TokenAndPositionEmbedd                                     
 ing)                                                            
                                                                 
 transformer_block_3 (Transf  (None, 100, 128)         198272    
 ormerBlock)                           

In [23]:
seed_text = "harry looked at ron"
generated_text = generate_text(
    seed_text,
    next_words=50,
    max_sequence_len=config.SEQ_LENGTH + 1
)

print(f"\nGenerated text length: {len(generated_text)} characters")
print(f"\nGenerated text:\n{generated_text}")  


Generated text length: 269 characters

Generated text:
harry looked at ron â€œyou donâ€™t think so was no one ever remembered that the last word harry had been watching the hat had bowed in case for the last shop on the other side three of them â€œmiss granger you foolish package just read by a bit of a week â€ said harry


## What Is Missing Compared to ChatGPT?

- **Masked Attention**  
  ChatGPT uses causal masking so that a word cannot see future words during training.  
  Our model uses regular attention, which allows it to see the entire sequence.

- **Multiple Stacked Transformer Blocks**  
  ChatGPT has many layers (e.g., 12, 24, 96 layers).  
  Our model has only one Transformer block.

- **Tokenization & Byte-Pair Encoding (BPE)**  
  ChatGPT does not use simple tokenization; it uses Byte-Pair Encoding (BPE) or WordPiece for better vocabulary handling.  
  Our model uses basic word tokenization.

- **Training on Large Datasets**  
  ChatGPT is trained on hundreds of GBs of text.  
  Our model is trained on a single Harry Potter book (very limited).

- **Decoding Strategies for Text Generation**  
  ChatGPT uses sampling (top-k, nucleus sampling) or beam search to generate text.  
  Our model does not have a decoding strategy.


### Key Transformer Concepts

- **Context window** → how much the model can see  
  (number of tokens available at once for attention)

- **Layers** → how deeply the model can reason  
  (number of Transformer blocks stacked on top of each other)

In simple terms:
- Increasing the **context window** improves memory
- Increasing the **number of layers** improves reasoning depth

Both must be balanced for an effective language model.
